<a href="https://colab.research.google.com/github/KacperLatecki/Zadania_ED/blob/main/Naive_Bayes_zadania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import LabelEncoder


# Zadanie 1:  

## Email Spam

 Masz dane o 12 emailach z informacją czy to spam czy nie:

 **Zadania do wykonania:**

**a) Ręczne obliczenia**
1. Oblicz prawdopodobieństwa a priori: P(Spam=TAK) i P(Spam=NIE)
2. Dla każdej cechy oblicz prawdopodobieństwa warunkowe
3. Przewidź klasę dla nowego emaila:
```
   Słowo_1 = 'darmowy'
   Słowo_2 = 'wygrana'  
   Wykrzyknik = 'TAK'
```

Oblicz prawdopodobieństwa dla obu klas (TAK lub NIE) i znormalizuj

**b) Implementacja w Python**

1. Zaimplementuj obliczenia z punktu a) w Python (bez sklearn)
2. Porównaj wyniki z ręcznymi obliczeniami

**c) Sklearn**

1. Użyj `CategoricalNB` z sklearn do wytrenowania modelu
2. Porównaj wyniki z własnymi obliczeniami
3. Wyjaśnij różnice (jeśli są)

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import LabelEncoder

# =========================
# NAIVE BAYES – ZADANIA (JEDEN BLOK)
# =========================

# --- Dane (Email Spam) ---
data_spam = {
    'Słowo_1': ['darmowy', 'darmowy', 'spotkanie', 'raport', 'oferta', 'darmowy',
                'spotkanie', 'oferta', 'raport', 'darmowy', 'spotkanie', 'oferta'],
    'Słowo_2': ['wygrana', 'wygrana', 'jutro', 'kwartalny', 'specjalna', 'rabat',
                'dziś', 'limitowana', 'miesięczny', 'rabat', 'pilne', 'wyjątkowa'],
    'Wykrzyknik': ['TAK', 'TAK', 'NIE', 'NIE', 'TAK', 'TAK',
                   'NIE', 'TAK', 'NIE', 'TAK', 'NIE', 'TAK'],
    'Spam': ['TAK', 'TAK', 'NIE', 'NIE', 'NIE', 'TAK',
             'NIE', 'NIE', 'NIE', 'TAK', 'NIE', 'NIE']
}

df = pd.DataFrame(data_spam)
print("Dane:")
display(df)

# --- Funkcja ręczna Naive Bayes (bez wygładzania) ---
def naive_bayes_predict_no_smoothing(df, X_new, target_col="Spam"):
    priors = df[target_col].value_counts(normalize=True)
    features = [c for c in df.columns if c != target_col]

    conditional = {}
    for cls in priors.index:
        subset = df[df[target_col] == cls]
        conditional[cls] = {}
        for col in features:
            conditional[cls][col] = subset[col].value_counts(normalize=True)

    raw = {}
    for cls in priors.index:
        p = priors[cls]
        for col in features:
            p *= conditional[cls][col].get(X_new[col], 0.0)
        raw[cls] = p

    total = sum(raw.values())
    post = {k: (v/total if total > 0 else 0.0) for k, v in raw.items()}
    return priors, conditional, raw, post

# --- Nowy email do klasyfikacji ---
new_mail = {'Słowo_1': 'darmowy', 'Słowo_2': 'wygrana', 'Wykrzyknik': 'TAK'}
print("\nNowy mail:", new_mail)

# --- a) Obliczenia ręczne ---
priors, conditional, raw, post = naive_bayes_predict_no_smoothing(df, new_mail, "Spam")

print("\nPrawdopodobieństwa a priori:")
display(priors)

print("\nPrawdopodobieństwa warunkowe P(cecha | klasa):")
for cls in conditional:
    print(f"\nKlasa = {cls}")
    for col in conditional[cls]:
        print(f"  {col}:")
        display(conditional[cls][col].to_frame("P"))

print("\nNienormalizowane P(klasa)*ΠP(cecha|klasa):", raw)
print("Znormalizowane (posteriori):", post)
print("➡️ Predykcja (ręcznie):", max(post, key=post.get) if sum(raw.values()) > 0 else "NIEOKREŚLONE (0-prob)")

# --- b) Sklearn: CategoricalNB (ma wygładzanie Laplace’a) ---
df_enc = df.copy()
encoders = {}

for col in df.columns:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df[col])
    encoders[col] = le

X = df_enc[['Słowo_1', 'Słowo_2', 'Wykrzyknik']]
y = df_enc['Spam']

model = CategoricalNB()
model.fit(X, y)

# przygotuj nowy mail do formatu liczbowego
new_mail_df = pd.DataFrame([new_mail])
for col in new_mail_df.columns:
    new_mail_df[col] = encoders[col].transform(new_mail_df[col])

proba = model.predict_proba(new_mail_df)[0]
class_names = encoders["Spam"].inverse_transform(model.classes_)

print("\nSklearn – prawdopodobieństwa:")
display(pd.DataFrame([proba], columns=class_names))

pred_sklearn = encoders["Spam"].inverse_transform(model.predict(new_mail_df))[0]
print("➡️ Predykcja (sklearn):", pred_sklearn)

print("\nWniosek:")
print("- Ręcznie (bez wygładzania) i sklearn (z Laplace) mogą dawać te same klasy,")
print("  ale wartości prawdopodobieństw mogą się różnić przez wygładzanie w sklearn.")


Dane:


,Słowo_1,Słowo_2,Wykrzyknik,Spam
0,darmowy,wygrana,TAK,TAK
1,darmowy,wygrana,TAK,TAK
2,spotkanie,jutro,NIE,NIE
3,raport,kwartalny,NIE,NIE
4,oferta,specjalna,TAK,NIE
5,darmowy,rabat,TAK,TAK
6,spotkanie,dziś,NIE,NIE
7,oferta,limitowana,TAK,NIE
8,raport,miesięczny,NIE,NIE
9,darmowy,rabat,TAK,TAK



Nowy mail: {'Słowo_1': 'darmowy', 'Słowo_2': 'wygrana', 'Wykrzyknik': 'TAK'}

Prawdopodobieństwa a priori:


,proportion
Spam,
NIE,0.666667
TAK,0.333333



Prawdopodobieństwa warunkowe P(cecha | klasa):

Klasa = NIE
  Słowo_1:


...

  Wykrzyknik:


,P
Wykrzyknik,
TAK,1.0



Nienormalizowane P(klasa)*ΠP(cecha|klasa): {'NIE': np.float64(0.0), 'TAK': np.float64(0.16666666666666666)}
Znormalizowane (posteriori): {'NIE': np.float64(0.0), 'TAK': np.float64(1.0)}
➡️ Predykcja (ręcznie): TAK

Sklearn – prawdopodobieństwa:


,NIE,TAK
0,0.032119,0.967881


➡️ Predykcja (sklearn): TAK

Wniosek:
- Ręcznie (bez wygładzania) i sklearn (z Laplace) mogą dawać te same klasy,
  ale wartości prawdopodobieństw mogą się różnić przez wygładzanie w sklearn.
